# FaceReader Emotion Pipeline — combined notebook

Runs the full Python side of the pipeline end to end, one timepoint at a time,
from the relabelling stage through to the analysis-ready files. Set every path
**once** in the Configuration cell below, then run the notebook top to bottom.

**Stages**
1. Extract event mapping from each FaceReader `.frx` project file
2. Relabel the raw frame-level export (splits multi-event athletes)
3. Inject the verified Win/Loss result and apply sample exclusions
4. Aggregate frames to one value per athlete-event
5. Convert to analysis-ready `Study1_Winners` / `Study2_Losers` files

The two R stages (`technical_validation.R`, `frame_retention_per_participant.R`)
are run separately after this notebook, on the analysis-ready files it produces.

**Inputs you provide** (raw / verified source files):
- one `.frx` project file per timepoint — Result has two (Part 1 + Part 2)
- one raw FaceReader detailed Excel export per timepoint
- one verified v6 reference file per timepoint (supplies the `Result` column)
- one master list (metadata; Result fallback)

Everything else is generated into the working folders defined below.


## Setup

In [ ]:
# Colab: mount Drive if your files live there. Local: skip this cell.
try:
   from google.colab import drive
   drive.mount('/content/drive')
except Exception:
   pass


Mounted at /content/drive


In [ ]:
import os, re, csv, zipfile
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd


## Configuration

Edit these paths, then run the whole notebook. `TIMEPOINTS` maps each timepoint
to its inputs. `Result` takes a list of two `.frx` files; every other timepoint
takes one. All generated files land under `WORK_DIR`; the final analysis-ready
files land in `OUT_DIR`.


In [ ]:
# ---- top-level folders ----
BASE = "/content/drive/MyDrive/Final FR Aggregator Check PY"

WORK_DIR = BASE                                    # intermediates (generated)
OUT_DIR  = BASE + "/Analysis_Ready"                # final analysis-ready files (subfolder keeps them separate)
MASTER_LIST_FILE = BASE + "/Master List_Final.xlsx"

TIMEPOINTS = {
    "Pre": {
        "frx": BASE + "/Frx outputs/PreMatch Batch Analysis Feb 18.frx",
        "raw": BASE + "/Raw Outputs/PreMatch Batch Analysis Feb 18_20260223_123748_detailed.xlsx",
        "v6":  BASE + "/v6 Processed files/Pre_PROCESSED_v6.xlsx",
    },
    "Mid": {
        "frx": BASE + "/Frx outputs/Midmatch - ALL Analyses.frx",
        "raw": BASE + "/Raw Outputs/Midmatch - ALL Analyses_20260714_113838_detailed.xlsx",
        "v6":  BASE + "/v6 Processed files/Mid_PROCESSED_v6.xlsx",
    },
    "Result": {
        "frx": [
            BASE + "/Frx outputs/Result  Batch Analysis - Feb 18.frx",
            BASE + "/Frx outputs/Result Part 2 Batch Analysis Feb 18.frx",
        ],
        "raw": BASE + "/Raw Outputs/Combined Result Batch Analysis - Jun 3_detailed.xlsx",
        "v6":  BASE + "/v6 Processed files/Result_PROCESSED_v6.xlsx",
    },
    "Post": {
        "frx": BASE + "/Frx outputs/Postmatch Batch Analysis Feb 18_BatchRecovery.frx",
        "raw": BASE + "/Raw Outputs/Postmatch Batch Analysis Jun 3_detailed.xlsx",
        "v6":  BASE + "/v6 Processed files/Post_PROCESSED_v6.xlsx",
    },
}

RESULT_KEEP_FROM_PART2 = {"565"}

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

def work(name):
    return os.path.join(WORK_DIR, name)


In [ ]:
import os
for tp, cfg in TIMEPOINTS.items():
    for key, val in cfg.items():
        paths = val if isinstance(val, list) else [val]
        for p in paths:
            print(("OK  " if os.path.exists(p) else "MISS") + f"  [{tp}/{key}] {p}")
for label, p in [("MASTER", MASTER_LIST_FILE), ("WORK", WORK_DIR), ("OUT", OUT_DIR)]:
    print(("OK  " if os.path.exists(p) else "MISS") + f"  [{label}] {p}")

OK    [Pre/frx] /content/drive/MyDrive/Final FR Aggregator Check PY/Frx outputs/PreMatch Batch Analysis Feb 18.frx
OK    [Pre/raw] /content/drive/MyDrive/Final FR Aggregator Check PY/Raw Outputs/PreMatch Batch Analysis Feb 18_20260223_123748_detailed.xlsx
OK    [Pre/v6] /content/drive/MyDrive/Final FR Aggregator Check PY/v6 Processed files/Pre_PROCESSED_v6.xlsx
OK    [Mid/frx] /content/drive/MyDrive/Final FR Aggregator Check PY/Frx outputs/Midmatch - ALL Analyses.frx
OK    [Mid/raw] /content/drive/MyDrive/Final FR Aggregator Check PY/Raw Outputs/Midmatch - ALL Analyses_20260714_113838_detailed.xlsx
OK    [Mid/v6] /content/drive/MyDrive/Final FR Aggregator Check PY/v6 Processed files/Mid_PROCESSED_v6.xlsx
OK    [Result/frx] /content/drive/MyDrive/Final FR Aggregator Check PY/Frx outputs/Result  Batch Analysis - Feb 18.frx
OK    [Result/frx] /content/drive/MyDrive/Final FR Aggregator Check PY/Frx outputs/Result Part 2 Batch Analysis Feb 18.frx
OK    [Result/raw] /content/drive/MyDrive/Fi

## Shared helpers

Key-matching normalisation shared across stages. Serials are stripped of leading
zeros before comparison because a zero-padded text serial (`"0099"`) silently
becomes the integer `99` on an Excel round-trip; without this, padded and
unpadded copies of the same serial would fail to match.


In [ ]:
def normalize_sr(x):
    """Leading integer run of a label, as a string ('0099_2Para24 - ...' -> '99')."""
    m = re.match(r"^0*(\d+)", str(x).strip())
    return m.group(1) if m else None

def normalize_ai(x):
    """Digits only from an Analysis Index ('Analysis 7', '7', 'AI7' -> '7')."""
    s = str(x)
    d = "".join(ch for ch in s if ch.isdigit())
    return d if d else s.strip()

def norm_key(label):
    """Everything before the first ' - ', lowercased; bare serials unpadded.
    Compound event labels (containing '_') are left as-is."""
    head = str(label).split(" - ")[0].strip("_ ")
    if re.fullmatch(r"0*\d+", head):
        return str(int(head))
    return head.lower()


## Stage 1 — Extract event mapping from the `.frx` project(s)

For each analysis in the project file, records the source video's folder name and
filename. Only the folder name and filename are used (never the drive letter or
full path), so the mapping is stable across machines. For `Result`, the two
project parts are concatenated; where a serial appears in both parts, the Part 2
rows are kept.


In [ ]:
def frx_rows(frx_path):
    z = zipfile.ZipFile(frx_path)
    if "Meta.xml" not in z.namelist():
        raise SystemExit(f"No Meta.xml in {frx_path}")
    root = ET.fromstring(z.read("Meta.xml"))
    rows = []
    for p in root.findall(".//Participant"):
        sr_no = (p.findtext("ParticipantInformation/ParticipantName") or "").strip()
        for a in p.findall("./Analyses/Analysis"):
            analysis_id = a.findtext("UniqueID") or ""
            src = a.findtext("./ImageSourceData/SourceFilenames/string") or ""
            parts = src.replace("/", "\\").split("\\")
            rows.append({
                "Sr_No": sr_no,
                "Analysis_Index": analysis_id,
                "Source_Folder_Name": parts[-2] if len(parts) >= 2 else "",
                "Source_Video_Filename": parts[-1] if parts else "",
                "Full_Source_Path": src,
            })
    return rows

def build_mapping(tp, cfg):
    frx = cfg["frx"]
    if isinstance(frx, str):
        rows = frx_rows(frx)
    else:
        # multi-part (Result): keep Part 2 for serials present in both parts
        part_rows = [frx_rows(f) for f in frx]
        later_serials = {normalize_sr(r["Sr_No"]) for r in part_rows[-1]}
        rows = []
        for i, pr in enumerate(part_rows):
            is_last = (i == len(part_rows) - 1)
            for r in pr:
                sr = normalize_sr(r["Sr_No"])
                if not is_last and sr in later_serials and sr in RESULT_KEEP_FROM_PART2:
                    continue  # drop earlier-part copy; keep Part 2 version
                rows.append(r)

    out_csv = work(f"{tp}_Analysis_To_Video_Mapping.csv")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["Sr_No","Analysis_Index","Source_Folder_Name",
                                          "Source_Video_Filename","Full_Source_Path"])
        w.writeheader(); w.writerows(rows)

    folders_by_sr = defaultdict(set)
    for r in rows:
        folders_by_sr[normalize_sr(r["Sr_No"])].add(r["Source_Folder_Name"])
    n_multi = sum(1 for v in folders_by_sr.values() if len(v) > 1)
    print(f"[{tp}] mapping rows: {len(rows)} | multi-event serials: {n_multi} -> {os.path.basename(out_csv)}")
    return out_csv


## Stage 2 — Relabel the raw frame-level export

Serials whose analyses span more than one source folder are multi-event athletes;
their `Participant Name` is replaced per row with the source folder name for that
analysis, so each event becomes a distinct participant. Single-event athletes are
untouched. A cross-check audit reports whether the split matches the expected
number of events per serial.


In [ ]:
def relabel(tp, mapping_csv, cfg):
    mapping = pd.read_csv(mapping_csv, dtype=str)
    mapping["Sr_No_Norm"] = mapping["Sr_No"].apply(normalize_sr)
    mapping["AI_Key"] = mapping["Analysis_Index"].apply(normalize_ai)

    folder_counts = mapping.groupby("Sr_No_Norm")["Source_Folder_Name"].nunique()
    dup = set(folder_counts[folder_counts > 1].index)

    dmap = mapping[mapping["Sr_No_Norm"].isin(dup)].copy()
    lookup = dmap.set_index(["Sr_No_Norm","AI_Key"])["Source_Folder_Name"]
    lookup = lookup[~lookup.index.duplicated(keep="first")]

    df = pd.read_excel(cfg["raw"], sheet_name=0)
    df.columns = [str(c).strip() for c in df.columns]
    if "Participant Name" not in df.columns or "Analysis Index" not in df.columns:
        raise SystemExit(f"[{tp}] raw export missing 'Participant Name'/'Analysis Index'")

    orig_pn = df["Participant Name"].astype(str).str.strip()
    pn_norm = df["Participant Name"].apply(normalize_sr)
    ai_key = df["Analysis Index"].apply(normalize_ai)
    is_dup = pn_norm.isin(dup)

    keys = list(zip(pn_norm[is_dup], ai_key[is_dup]))
    new_labels = lookup.reindex(keys)
    if int(new_labels.isna().sum()):
        miss = sorted(set(pn_norm[is_dup][new_labels.isna().values]))
        print(f"[{tp}] WARNING: {int(new_labels.isna().sum())} dup row(s) unmatched, kept as-is. Serials: {', '.join(miss)}")

    final = new_labels.where(new_labels.notna(),
                             pd.Series(orig_pn[is_dup].values, index=new_labels.index))
    df["Participant Name"] = df["Participant Name"].astype(object)
    df.loc[is_dup, "Participant Name"] = final.values

    out = work(f"{tp}_Analysis_RELABELED.xlsx")
    df.to_excel(out, index=False)
    print(f"[{tp}] relabel: {orig_pn.nunique()} -> {df['Participant Name'].astype(str).str.strip().nunique()} labels -> {os.path.basename(out)}")
    return out


## Stage 3 — Inject the verified Win/Loss result and apply exclusions

Writes the verified `Result` onto every row, joined per (serial, event) from the
v6 reference. Rows with no v6 match (athletes outside the analysed sample) are
dropped. Fails loudly on a blank result or on colliding keys that disagree.


In [ ]:
def inject(tp, relabeled_path, cfg):
    v6 = pd.read_excel(cfg["v6"], sheet_name="Pooled_By_Participant")
    for col in ("Sr_No","Participant_Name_Full","Result"):
        if col not in v6.columns:
            raise SystemExit(f"[{tp}] v6 missing column: {col}")
    v6["Sr_Norm"] = v6["Sr_No"].apply(normalize_sr)
    v6["Key"] = v6["Participant_Name_Full"].apply(norm_key)

    blank = v6["Result"].isna() | (v6["Result"].astype(str).str.strip() == "")
    if blank.any():
        raise SystemExit(f"[{tp}] v6 has blank Result for Sr_No {sorted(v6.loc[blank,'Sr_No'].unique())}")

    lookup = v6.set_index(["Sr_Norm","Key"])["Result"]
    if lookup.index.duplicated().any():
        collide = v6[v6.duplicated(["Sr_Norm","Key"], keep=False)]
        conflict = collide.groupby(["Sr_Norm","Key"])["Result"].nunique()
        bad = conflict[conflict > 1]
        if len(bad):
            raise SystemExit(f"[{tp}] colliding (serial,event) keys disagree on Result:\n"
                             f"{collide[['Sr_No','Participant_Name_Full','Result']]}")
        lookup = lookup[~lookup.index.duplicated(keep="first")]

    df = pd.read_excel(relabeled_path, sheet_name=0)
    df.columns = [str(c).strip() for c in df.columns]
    pn = df["Participant Name"].astype(str).str.strip()
    keys = pd.MultiIndex.from_arrays([pn.apply(normalize_sr), pn.apply(norm_key)])
    df["Result"] = lookup.reindex(keys).values

    missing = df["Result"].isna()
    df = df[~missing].copy()
    if df["Result"].isna().any():
        raise SystemExit(f"[{tp}] blank Result survived filtering")

    out = work(f"{tp}_Analysis_RELABELED_FINAL.xlsx")
    df.to_excel(out, index=False)
    print(f"[{tp}] inject: dropped {int(missing.sum())} unmatched row(s) | retained {len(df)} -> {os.path.basename(out)}")
    return out


## Stage 4 — Aggregate to one value per athlete-event

Groups strictly by the full participant label. Error tokens become missing; each
measure is the mean over valid frames pooled across a participant's analyses.
Result comes from the injected column (majority if rows disagree), else the master
list; other metadata from the master list. Writes `Pooled_By_Participant`,
`Analysis_Quality`, `Verification`.


In [ ]:
EMOTION_COLUMNS = ["Neutral","Happy","Sad","Angry","Surprised","Scared","Disgusted","Valence","Arousal"]
AU_PREFIX = "Action Unit"
ERROR_TOKENS = {"fit_failed","find_failed","no_face","missing","n/a","na","error","invalid","none",""}
INVALID_30, INVALID_50 = 0.30, 0.50
PD_MAP = {"high":"High","hi":"High","h":"High","highpd":"High","high pd":"High","high-pd":"High",
          "low":"Low","lo":"Low","l":"Low","lowpd":"Low"}
GENDER_MAP = {"male":"Male","m":"Male","man":"Male","female":"Female","f":"Female","woman":"Female"}
RESULT_MAP = {"win":"Win","won":"Win","w":"Win","winner":"Win","gold":"Win",
              "loss":"Loss","lost":"Loss","l":"Loss","lose":"Loss","loser":"Loss","4th":"Loss"}

def _coerce(series):
    def one(v):
        s = str(v).strip().lower()
        if s in ERROR_TOKENS: return np.nan
        try: return float(v)
        except (ValueError, TypeError): return np.nan
    return series.map(one)

def _sr(label):
    m = re.match(r"^0*(\d+)", str(label).strip())
    return int(m.group(1)) if m else None

def _std(v, m):
    if pd.isna(v): return None
    return m.get(str(v).strip().lower(), str(v).strip())

def _comp(v):
    if pd.isna(v): return None
    vl = str(v).strip().lower()
    if vl.startswith("para"): return "Paralympic"
    if vl.startswith("oly") or "olympic" in vl: return "Olympic"
    return str(v).strip()

def load_master(path):
    if not path or not os.path.exists(path):
        print("Master list: not used"); return {}
    m = pd.read_excel(path); m.columns = [str(c).strip() for c in m.columns]
    def find(*keys):
        for c in m.columns:
            if any(k in c.lower() for k in keys): return c
        return None
    cols = dict(sr=find("serial","serail","sr no","sr_no","srno"), name=find("athlete","name"),
                nat=find("nation","country"), pd=find("pd","pressure"), gen=find("gender","sex"),
                comp=find("olympic","paralympic","compet"), res=find("win","lose","result","outcome"))
    out = {}
    for _, r in m.iterrows():
        sr = _sr(r[cols["sr"]]) if cols["sr"] else None
        if sr is None: continue
        out[sr] = {"Name": r[cols["name"]] if cols["name"] else None,
                   "Nationality": r[cols["nat"]] if cols["nat"] else None,
                   "PD": _std(r[cols["pd"]], PD_MAP) if cols["pd"] else None,
                   "Gender": _std(r[cols["gen"]], GENDER_MAP) if cols["gen"] else None,
                   "Competition": _comp(r[cols["comp"]]) if cols["comp"] else None,
                   "Result": _std(r[cols["res"]], RESULT_MAP) if cols["res"] else None}
    print(f"Master list: {len(out)} athletes")
    return out

def aggregate(tp, injected_path, master):
    df = pd.read_excel(injected_path); df.columns = [str(c).strip() for c in df.columns]
    pn_col, ai_col = "Participant Name", "Analysis Index"
    target = [c for c in df.columns if c in EMOTION_COLUMNS or c.startswith(AU_PREFIX)]
    emo_present = [c for c in EMOTION_COLUMNS if c in target]
    for c in target: df[c] = _coerce(df[c])
    ref = "Neutral" if "Neutral" in target else target[0]
    has_res = "Result" in df.columns

    pooled, aq = [], []
    for pn, grp in df.groupby(pn_col, sort=False):
        sr = _sr(pn); meta = master.get(sr, {}) if sr is not None else {}
        result = meta.get("Result")
        if has_res:
            inj = grp["Result"].dropna(); inj = inj[inj.astype(str).str.strip() != ""]
            if len(inj):
                uq = inj.map(lambda v: _std(v, RESULT_MAP)).unique()
                result = Counter(inj.map(lambda v: _std(v, RESULT_MAP))).most_common(1)[0][0] if len(uq) > 1 else uq[0]
        n_an = grp[ai_col].nunique() if ai_col in grp.columns else 1
        details = []
        for ai, asub in (grp.groupby(ai_col, sort=True) if ai_col in grp.columns else [("Analysis 1", grp)]):
            tot = len(asub); val = int(asub[ref].notna().sum()); inv = tot - val
            ipct = round(inv/tot*100,1) if tot else 0.0
            details.append(f"AI{ai}: {val}v/{tot}t ({ipct}% inv)")
            aq.append({"Sr_No":sr,"Duplicate_Group":"","Participant_Name_Full":pn,"Name":meta.get("Name"),
                       "PD":meta.get("PD"),"Gender":meta.get("Gender"),"Competition":meta.get("Competition"),
                       "Result":result,"Analysis_Index":ai,"Total_Frames":tot,"Valid_Frames":val,
                       "Invalid_Frames":inv,"Invalid_Pct":ipct,
                       "Flagged_30pct":"YES" if (tot and inv/tot>INVALID_30) else "no",
                       "Flagged_50pct":"YES" if (tot and inv/tot>INVALID_50) else "no"})
        tot = len(grp); val = int(grp[ref].notna().sum())
        row = {"Sr_No":sr,"Duplicate_Group":"","Name":meta.get("Name"),"Participant_Name_Full":pn,
               "Nationality":meta.get("Nationality"),"PD":meta.get("PD"),"Gender":meta.get("Gender"),
               "Competition":meta.get("Competition"),"Result":result,"TimePoint":tp,"Num_Analyses":n_an,
               "Analyses":", ".join(str(x) for x in sorted(grp[ai_col].unique())) if ai_col in grp.columns else "",
               "Total_Frames":tot,"Valid_Frames":val,"Invalid_Frames":tot-val,
               "Valid_Pct":round(val/tot*100,1) if tot else 0.0,"Analysis_Quality_Detail":" | ".join(details)}
        for col in target:
            vv = grp[col].dropna()
            row[f"{col}_AllAnalyses"] = vv.mean() if len(vv) else np.nan
            row[f"{col}_AllAnalyses_NFrames"] = len(vv)
            row[f"{col}_Sum"] = vv.sum(); row[f"{col}_Count"] = len(vv)
        pooled.append(row)

    P = pd.DataFrame(pooled)
    sc = P["Sr_No"].value_counts(); multi = set(sc[sc>1].index)
    P["Duplicate_Group"] = P["Sr_No"].map(lambda s: str(int(s)) if s in multi else "")
    A = pd.DataFrame(aq)
    A["Duplicate_Group"] = A["Participant_Name_Full"].map(
        P.set_index("Participant_Name_Full")["Duplicate_Group"].to_dict()).fillna("")

    meta_cols = ["Sr_No","Duplicate_Group","Name","Participant_Name_Full","Nationality","PD","Gender",
                 "Competition","Result","TimePoint","Num_Analyses","Analyses","Total_Frames","Valid_Frames",
                 "Invalid_Frames","Valid_Pct","Analysis_Quality_Detail"]
    ordered = [c for c in EMOTION_COLUMNS if c in target] + sorted([c for c in target if c.startswith(AU_PREFIX)])
    measure_cols = []
    for col in ordered: measure_cols += [f"{col}_AllAnalyses", f"{col}_AllAnalyses_NFrames"]
    pooled_out = P[meta_cols+measure_cols].sort_values(["Sr_No","Participant_Name_Full"]).reset_index(drop=True)

    aq_cols = ["Sr_No","Duplicate_Group","Participant_Name_Full","Name","PD","Gender","Competition","Result",
               "Analysis_Index","Total_Frames","Valid_Frames","Invalid_Frames","Invalid_Pct","Flagged_30pct","Flagged_50pct"]
    aq_out = A[aq_cols].sort_values(["Sr_No","Analysis_Index"]).reset_index(drop=True)

    ver_cols = ["Sr_No","Name","Valid_Frames"]
    for col in emo_present[:3]:
        for s in (f"{col}_AllAnalyses", f"{col}_Sum", f"{col}_Count"):
            if s in P.columns: ver_cols.append(s)
    ver_out = P[ver_cols].copy()

    out = work(f"{tp}_PROCESSED_v9_PY.xlsx")
    with pd.ExcelWriter(out, engine="openpyxl") as xw:
        pooled_out.to_excel(xw, sheet_name="Pooled_By_Participant", index=False)
        aq_out.to_excel(xw, sheet_name="Analysis_Quality", index=False)
        ver_out.to_excel(xw, sheet_name="Verification", index=False)
    print(f"[{tp}] aggregate: {len(pooled_out)} participants -> {os.path.basename(out)}")
    return out


## Stage 5 — Convert to analysis-ready `Study1_Winners` / `Study2_Losers`

Keeps the 9 emotions + 20 combined AUs (left/right dropped), renames to the short
codes the R analysis expects, and derives `Vision` (Olympic -> Sighted, everything
else -> Blind). Splits Winners / Losers into two sheets per timepoint.

`Blindness Status` is written as a placeholder (`N/A (Sighted)` / `Unknown`): the
analysis groups only by Vision x PD and never reads this column, so it does not
affect any result.


In [ ]:
EMOTIONS = ["Neutral","Happy","Sad","Angry","Surprised","Scared","Disgusted","Valence","Arousal"]
AU_MAP = {
    "Action Unit 01 - Inner Brow Raiser":"AU01_InnerBrowRaiser","Action Unit 02 - Outer Brow Raiser":"AU02_OuterBrowRaiser",
    "Action Unit 04 - Brow Lowerer":"AU04_BrowLowerer","Action Unit 05 - Upper Lid Raiser":"AU05_UpperLidRaiser",
    "Action Unit 06 - Cheek Raiser":"AU06_CheekRaiser","Action Unit 07 - Lid Tightener":"AU07_LidTightener",
    "Action Unit 09 - Nose Wrinkler":"AU09_NoseWrinkler","Action Unit 10 - Upper Lip Raiser":"AU10_UpperLipRaiser",
    "Action Unit 12 - Lip Corner Puller":"AU12_LipCornerPuller","Action Unit 14 - Dimpler":"AU14_Dimpler",
    "Action Unit 15 - Lip Corner Depressor":"AU15_LipCornerDepressor","Action Unit 17 - Chin Raiser":"AU17_ChinRaiser",
    "Action Unit 18 - Lip Pucker":"AU18_LipPucker","Action Unit 20 - Lip Stretcher":"AU20_LipStretcher",
    "Action Unit 23 - Lip Tightener":"AU23_LipTightener","Action Unit 24 - Lip Pressor":"AU24_LipPressor",
    "Action Unit 25 - Lips Part":"AU25_LipsPart","Action Unit 26 - Jaw Drop":"AU26_JawDrop",
    "Action Unit 27 - Mouth Stretch":"AU27_MouthStretch","Action Unit 43 - Eye Closure":"AU43_EyeClosure",
}
META_ORDER = ["Study","Timepoint","Sr_No","Participant_Name_Full","Name","Nationality","Gender",
              "Competition","Vision","Blindness Status","PD","Result"]

def _vision(comp):
    if pd.isna(comp): return None
    return "Sighted" if str(comp).strip().lower() == "olympic" else "Blind"

def make_ready(tp, pooled_path):
    df = pd.read_excel(pooled_path, sheet_name="Pooled_By_Participant")
    out = pd.DataFrame()
    out["Study"] = None
    out["Timepoint"] = tp
    for c in ["Sr_No","Participant_Name_Full","Name","Nationality","Gender","Competition","PD","Result"]:
        out[c] = df[c] if c in df.columns else None
    out["Vision"] = df["Competition"].map(_vision) if "Competition" in df.columns else None
    out["Blindness Status"] = out["Vision"].map(lambda v: "N/A (Sighted)" if v == "Sighted" else "Unknown")
    for e in EMOTIONS:
        col = f"{e}_AllAnalyses"; out[e] = df[col] if col in df.columns else None
    for long_name, short in AU_MAP.items():
        col = f"{long_name}_AllAnalyses"; out[short] = df[col] if col in df.columns else None
    ordered = META_ORDER + EMOTIONS + list(AU_MAP.values())
    out = out[[c for c in ordered if c in out.columns]]

    winners = out[out["Result"] == "Win"].copy();  winners["Study"] = "Study 1"
    losers  = out[out["Result"] == "Loss"].copy(); losers["Study"]  = "Study 2"
    dst = os.path.join(OUT_DIR, f"{tp}_Analysed_Participants.xlsx")
    with pd.ExcelWriter(dst, engine="openpyxl") as xw:
        winners.to_excel(xw, sheet_name="Study1_Winners", index=False)
        losers.to_excel(xw, sheet_name="Study2_Losers", index=False)
    other = out[~out["Result"].isin(["Win","Loss"])]
    msg = f"[{tp}] analysis-ready: Winners {len(winners)} | Losers {len(losers)}"
    if len(other): msg += f" | EXCLUDED {len(other)} non Win/Loss: {sorted(other['Sr_No'].dropna().astype(int).unique())}"
    print(msg + f" -> {os.path.basename(dst)}")
    return dst


## Run the whole pipeline

Runs all five stages for every timepoint in order. Watch the printed
warnings/exclusions between stages; anything unexpected there is worth stopping
for before trusting the analysis-ready files.


In [ ]:
master = load_master(MASTER_LIST_FILE)

for tp, cfg in TIMEPOINTS.items():
    print(f"\n===== {tp} =====")
    mapping_csv   = build_mapping(tp, cfg)
    relabeled     = relabel(tp, mapping_csv, cfg)
    injected      = inject(tp, relabeled, cfg)
    pooled        = aggregate(tp, injected, master)
    make_ready(tp, pooled)

print("\nDone. Analysis-ready files in:", OUT_DIR)
print("Next: run technical_validation.R and frame_retention_per_participant.R on these files.")


Master list: 617 athletes

===== Pre =====
[Pre] mapping rows: 1372 | multi-event serials: 54 -> Pre_Analysis_To_Video_Mapping.csv
[Pre] relabel: 592 -> 659 labels -> Pre_Analysis_RELABELED.xlsx
[Pre] inject: dropped 5649 unmatched row(s) | retained 34408 -> Pre_Analysis_RELABELED_FINAL.xlsx
[Pre] aggregate: 577 participants -> Pre_PROCESSED_v9_PY.xlsx
[Pre] analysis-ready: Winners 290 | Losers 287 -> Pre_Analysed_Participants.xlsx

===== Mid =====
[Mid] mapping rows: 1652 | multi-event serials: 25 -> Mid_Analysis_To_Video_Mapping.csv
[Mid] relabel: 412 -> 444 labels -> Mid_Analysis_RELABELED.xlsx
[Mid] inject: dropped 3058 unmatched row(s) | retained 39382 -> Mid_Analysis_RELABELED_FINAL.xlsx
[Mid] aggregate: 411 participants -> Mid_PROCESSED_v9_PY.xlsx
[Mid] analysis-ready: Winners 212 | Losers 199 -> Mid_Analysed_Participants.xlsx

===== Result =====
[Result] mapping rows: 583 | multi-event serials: 17 -> Result_Analysis_To_Video_Mapping.csv
[Result] relabel: 344 -> 364 labels -> Re